# Weaviate Vector Database & Retriever Configuration | Gemini Embedding Model

In [15]:
import json
import os
import time

import dotenv
import weaviate
from google.genai import types, errors, Client
from weaviate.classes.config import Property, DataType, Configure
from weaviate.classes.query import MetadataQuery

In [14]:
client = weaviate.connect_to_custom(
    http_host="localhost",
    http_port=8080,
    http_secure=False,
    grpc_host="localhost",
    grpc_port=50051,
    grpc_secure=False,
)

try:
    client.collections.create(
        name="FAI_Guide",
        description="A collection of student guide information",
        properties=[
            Property(name="chunk_id", data_type=DataType.TEXT),
            Property(name="page_number", data_type=DataType.INT),
            Property(name="main", data_type=DataType.TEXT),
            Property(name="content", data_type=DataType.TEXT),
        ],
        vector_config=Configure.Vectors.self_provided(),
    )
    print("""\x1b[32;47mSuccess Mesage:\n--> Collection created successfully!\x1b[0m""")
except:
    print(
        """\x1b[30;41mInternal Error:\n--> The Weaviate VectorDB already exists.\x1b[0m"""
    )
finally:
    client.close()

Internal Error:
--> The Weaviate VectorDB already exists.


In [16]:
dotenv.load_dotenv()

gclient = Client(api_key=os.environ["GEMINI_API_KEY"])


def embed(texts, task, batch=10, pause=4):
    out = []
    for i in range(0, len(texts), batch):
        chunk = texts[i : i + batch]
        for attempt in range(6):
            try:
                res = gclient.models.embed_content(
                    model="gemini-embedding-001",
                    contents=chunk,
                    config=types.EmbedContentConfig(
                        task_type=task, output_dimensionality=768
                    ),
                )
                out.extend(e.values for e in res.embeddings)
                break
            except errors.ClientError as e:
                if e.code == 429:
                    time.sleep(20 * (attempt + 1))
                else:
                    raise
        else:
            raise RuntimeError(f"Gave up at batch starting {i}")
        time.sleep(pause)
    return out

In [ ]:
with open("semantic_chunks.json", "r+") as data:
    chunks = json.load(data)

texts = [f'{c["main"]}\n{c["content"]}' for c in chunks]
vecs = embed(texts, "RETRIEVAL_DOCUMENT")

In [9]:
client.connect()
col = client.collections.get("FAI_Guide")

try:
    assert len(chunks) == len(vecs), (len(chunks), len(vecs))

    with col.batch.dynamic() as batch:
        for c, v in zip(chunks, vecs):
            batch.add_object(
                properties={
                    "chunk_id": str(c["chunk_id"]),
                    "page_number": int(c["page_number"]),
                    "main": c["main"],
                    "content": c["content"],
                },
                vector=v,
            )

    failed = col.batch.failed_objects
    if failed:
        print(f"Failed: {len(failed)}")
        print(failed[0])

    print("Count:", col.aggregate.over_all(total_count=True).total_count)
finally:
    client.close()

Count: 149


In [19]:
client.connect()
col = client.collections.get("FAI_Guide")  # re-get after reconnect

q = "موقف التجنيدي للطلبة؟"
qv = embed([q], "RETRIEVAL_QUERY")[0]

res = col.query.hybrid(
    query=q,
    vector=qv,
    alpha=0.5,
    limit=3,
    return_metadata=MetadataQuery(score=True, certainty=True),
)

for o in res.objects:
    p = o.properties
    print(f"score={o.metadata.score:.3f} | page {p['page_number']} | {p['main']}")
    print(p["content"][:300])
    print("-" * 60)

score=0.500 | page 78 | إجراءات التجنيد
دليل الطالب: الاصدار الثا لث العام الجامعي2024 / 2025 79 كلية الذكاء الاصطناع ى- جامعة المنوفية 79  السجن أو التحفظ أو الاعتقال  تأخر قبول أو وصول ملفات الطلاب إجراءات التجنيد عند بلوغ الطالب سن الثمانية عشر من عمره عليه التوجه إلى مندوب التجنيد بالقسم أو البندر أو المركز الذي استخرج منه البطاقة ا
------------------------------------------------------------
score=0.252 | page 76 | إجراءات القيد والأوراق المطلوبة
دليل الطالب: الاصدار الثا لث العام الجامعي2024 / 2025 77 كلية الذكاء الاصطناع ى- جامعة المنوفية 77 3 - اصل شهادة الميلاد 4 - عدد10 صور شخصية4 * 6 5 - نموذج2جند من مكتب الاتصال العسكري بكلية الهندسة 6 - صورة من البطاقة الشخصية 7 - صور من بطاقة تلقي لقاح فيروس كورونا المستجد 8 - استمارة الكشف الطبي 9 
------------------------------------------------------------
score=0.244 | page 77 | مجلس الكلية واللوائح المنظمة
دليل الطالب: الاصدار الثا لث العام الجامعي2024 / 2025 78 كلية الذكاء الاصطناع ى- جامعة المنوفية 78 مجلس الكلية، ويجوز في حالة ا